In [ ]:
import subprocess
import sys
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
    'huggingface-hub>=0.26.0',
    'python-dotenv>=1.0.0',
    'pyyaml>=6.0',
    'requests>=2.32.0',
], check=True)


In [ ]:
import os
import json
import time
import threading
from pathlib import Path
from datetime import datetime

import yaml
import requests
from huggingface_hub import HfApi, CommitOperationAdd

WORK_DIR        = Path('/kaggle/working')
TOKENS_DIR      = WORK_DIR / 'tokens'
CHECKPOINT_PATH = WORK_DIR / 'checkpoint_p2c.json'
CONFIG_DIR      = Path('/kaggle/input/urdu-asr-pipelines/config')

WAVE_SIZE_BYTES = 500 * 1024 * 1024
BATCH_SIZE      = 50
MAX_RETRIES     = 12
COMMIT_DELAY    = 3.0

In [ ]:
def load_secrets():
    try:
        from kaggle_secrets import UserSecretsClient
        c = UserSecretsClient()
        secrets = {
            'HF_TOKEN_PRIMARY':   c.get_secret('HF_TOKEN_PRIMARY'),
            'HF_TOKEN_SECONDARY': c.get_secret('HF_TOKEN_SECONDARY'),
            'HF_TOKEN_TERTIARY':  c.get_secret('HF_TOKEN_TERTIARY'),
            'ANTHROPIC_API_KEY':  c.get_secret('ANTHROPIC_API_KEY'),
        }
        print('[secrets] loaded from Kaggle Secrets')
        return secrets
    except Exception:
        pass
    env_file = Path('.env')
    if env_file.exists():
        from dotenv import load_dotenv
        load_dotenv(env_file)
        print('[secrets] loaded from .env')
    required = ['HF_TOKEN_PRIMARY', 'HF_TOKEN_SECONDARY', 'HF_TOKEN_TERTIARY', 'ANTHROPIC_API_KEY']
    missing = [k for k in required if not os.environ.get(k)]
    if missing:
        raise RuntimeError(f'Missing secrets: {missing}')
    return {k: os.environ[k] for k in required}

SECRETS     = load_secrets()
HF_TOKEN    = SECRETS['HF_TOKEN_PRIMARY']

with open(CONFIG_DIR / 'hf_repos.yaml') as f:
    repos_cfg = yaml.safe_load(f)

STAGE1_REPO = repos_cfg['repos']['stage1_ce']['repo_id']
HF_API      = HfApi(token=HF_TOKEN)
print(f'[config] stage1 repo: {STAGE1_REPO}')

In [ ]:
def load_checkpoint():
    if CHECKPOINT_PATH.exists():
        try:
            with open(CHECKPOINT_PATH) as f:
                state = json.load(f)
            print(f'[checkpoint] local — uploaded={state["stats"]["uploaded"]} waves={state["stats"]["waves"]}')
            return state
        except Exception:
            pass
    print('[checkpoint] fresh start')
    return {
        'uploaded_ids': [],
        'failed_ids': [],
        'labels_uploaded': False,
        'stats': {'uploaded': 0, 'failed': 0, 'waves': 0},
        'last_updated': None,
    }


cp_lock = threading.Lock()

def save_checkpoint(state, upload=False):
    with cp_lock:
        state['last_updated'] = datetime.utcnow().strftime('%Y-%m-%dT%H:%M:%SZ')
        tmp = str(CHECKPOINT_PATH) + '.tmp'
        with open(tmp, 'w') as f:
            json.dump(state, f)
        os.replace(tmp, str(CHECKPOINT_PATH))
    if not upload:
        return
    for attempt in range(6):
        try:
            HF_API.upload_file(
                path_or_fileobj=json.dumps(state).encode(),
                path_in_repo='checkpoint_p2c.json',
                repo_id=STAGE1_REPO,
                repo_type='dataset',
                commit_message='p2c checkpoint',
            )
            return
        except Exception:
            time.sleep(min(2 ** attempt, 60))


state        = load_checkpoint()
uploaded_set = set(state['uploaded_ids'])

In [ ]:
def commit_batch(batch, wave_num, batch_idx, total_batches):
    ops = [
        CommitOperationAdd(
            path_in_repo=f'tokens/{p.name}',
            path_or_fileobj=str(p),
        )
        for p in batch if p.exists()
    ]
    if not ops:
        return True
    for attempt in range(MAX_RETRIES):
        try:
            HF_API.create_commit(
                repo_id=STAGE1_REPO,
                repo_type='dataset',
                commit_message=f'tokens wave {wave_num} batch {batch_idx}/{total_batches}',
                operations=ops,
            )
            time.sleep(COMMIT_DELAY)
            return True
        except Exception as e:
            if attempt == MAX_RETRIES - 1:
                print(f'  [commit] wave {wave_num} batch {batch_idx} FAILED: {e}')
                return False
            time.sleep(min(2 ** attempt, 120))
    return False


def flush_wave(wave_files, wave_num):
    existing   = [p for p in wave_files if p.exists()]
    size_mb    = sum(p.stat().st_size for p in existing) / 1024 / 1024
    print(f'\n[wave {wave_num}] {len(existing)} files ({size_mb:.1f} MB)')
    batches    = [existing[i:i + BATCH_SIZE] for i in range(0, len(existing), BATCH_SIZE)]
    committed  = 0
    for idx, batch in enumerate(batches):
        ok = commit_batch(batch, wave_num, idx + 1, len(batches))
        if ok:
            committed += len(batch)
            with cp_lock:
                for p in batch:
                    uploaded_set.add(p.stem)
                    state['uploaded_ids'].append(p.stem)
                state['stats']['uploaded'] += len(batch)
            print(f'  batch {idx+1}/{len(batches)} OK')
        else:
            with cp_lock:
                for p in batch:
                    state['failed_ids'].append(p.stem)
                state['stats']['failed'] += len(batch)
    with cp_lock:
        state['stats']['waves'] += 1
    for p in existing:
        p.unlink(missing_ok=True)
    save_checkpoint(state, upload=True)
    print(f'[wave {wave_num}] done — {committed}/{len(existing)} committed\n')


all_npy   = sorted(TOKENS_DIR.glob('*.npy'))
pending   = [p for p in all_npy if p.stem not in uploaded_set]
total_gb  = sum(p.stat().st_size for p in pending) / 1024**3

print(f'[p2c] {len(all_npy)} total .npy, {len(uploaded_set)} uploaded, {len(pending)} pending ({total_gb:.2f} GB)')

wave_buf   = []
wave_bytes = 0
wave_num   = state['stats']['waves'] + 1

for npy_path in pending:
    wave_buf.append(npy_path)
    wave_bytes += npy_path.stat().st_size
    if wave_bytes >= WAVE_SIZE_BYTES:
        flush_wave(wave_buf, wave_num)
        wave_buf   = []
        wave_bytes = 0
        wave_num  += 1

if wave_buf:
    flush_wave(wave_buf, wave_num)

print('[p2c] all token files uploaded')

In [ ]:
labels_path = WORK_DIR / 'intent_labels_with_shapes.jsonl'
if not labels_path.exists():
    labels_path = WORK_DIR / 'intent_labels.jsonl'

if labels_path.exists() and not state['labels_uploaded']:
    print('[labels] uploading intent_labels.jsonl...')
    for attempt in range(MAX_RETRIES):
        try:
            HF_API.upload_file(
                path_or_fileobj=str(labels_path),
                path_in_repo='intent_labels.jsonl',
                repo_id=STAGE1_REPO,
                repo_type='dataset',
                commit_message='intent_labels.jsonl — CE training labels',
            )
            state['labels_uploaded'] = True
            save_checkpoint(state, upload=True)
            print('[labels] uploaded successfully')
            break
        except Exception as e:
            time.sleep(min(2 ** attempt, 120))
elif state['labels_uploaded']:
    print('[labels] already uploaded')

print('\n[p2c] summary')
print(f'  uploaded .npy   : {state["stats"]["uploaded"]}')
print(f'  failed .npy     : {state["stats"]["failed"]}')
print(f'  waves committed : {state["stats"]["waves"]}')
print(f'  labels uploaded : {state["labels_uploaded"]}')
print(f'\n[done] stage1 repo: https://huggingface.co/datasets/{STAGE1_REPO}')
print('[done] pipeline 2 complete — ready for pipeline 3 (p3a_generate.ipynb)')